# 🦅 Aerial Object Classification — Exploratory Data Analysis

**Notebook outline:**
1. Dataset inspection & class distribution
2. Sample image visualisation
3. Pixel statistics (mean, std, channel distributions)
4. Image size distribution
5. Augmentation preview
6. Quick baseline model sanity check

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # project root

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# Project modules
from src.preprocess import (
    check_class_balance, visualize_samples,
    visualize_augmentations, get_data_generators
)
from src.utils import set_seeds, print_hardware_info

set_seeds(42)
print_hardware_info()

DATASET_ROOT = '../path/to/classification_dataset'  # ← SET THIS
CLASSES = ['bird', 'drone']

## 1. Class Distribution

In [ ]:
report = check_class_balance(DATASET_ROOT)

# Bar chart
splits = list(report.keys())
x = range(len(CLASSES))
fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
colors = ['#58a6ff', '#3fb950', '#d29922']
width = 0.25
for i, (split, counts) in enumerate(report.items()):
    vals = [counts.get(c, 0) for c in CLASSES]
    ax.bar([xi + i*width for xi in x], vals, width, label=split, color=colors[i], alpha=0.85)
ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels([c.upper() for c in CLASSES], color='#8b949e')
ax.set_ylabel('Image Count', color='#8b949e')
ax.set_title('Class Distribution per Split', color='white', fontsize=13)
ax.legend(facecolor='#21262d', labelcolor='white')
ax.tick_params(colors='#8b949e')
ax.spines[:].set_color('#30363d')
plt.tight_layout()
plt.show()

## 2. Sample Images

In [ ]:
visualize_samples(DATASET_ROOT, n=12)

## 3. Pixel Statistics

In [ ]:
from src.preprocess import IMG_SIZE

means, stds = {c: [] for c in CLASSES}, {c: [] for c in CLASSES}

for cls in CLASSES:
    folder = Path(DATASET_ROOT) / 'TRAIN' / cls
    for p in tqdm(list(folder.glob('*.jpg'))[:200], desc=cls):  # sample 200
        img = np.array(Image.open(p).convert('RGB').resize(IMG_SIZE)) / 255.0
        means[cls].append(img.mean(axis=(0,1)))
        stds[cls].append(img.std(axis=(0,1)))

for cls in CLASSES:
    m = np.mean(means[cls], axis=0)
    s = np.mean(stds[cls],  axis=0)
    print(f'{cls.upper():6s}  mean=[{m[0]:.3f}, {m[1]:.3f}, {m[2]:.3f}]'
          f'  std=[{s[0]:.3f}, {s[1]:.3f}, {s[2]:.3f}]')

## 4. Image Size Distribution

In [ ]:
widths, heights = [], []
for cls in CLASSES:
    for p in (Path(DATASET_ROOT) / 'TRAIN' / cls).glob('*.jpg'):
        w, h = Image.open(p).size
        widths.append(w)
        heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0d1117')
for ax, data, label, color in [
    (axes[0], widths,  'Width',  '#58a6ff'),
    (axes[1], heights, 'Height', '#3fb950'),
]:
    ax.set_facecolor('#161b22')
    ax.hist(data, bins=40, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(np.mean(data), color='white', ls='--', lw=1.5, label=f'Mean={np.mean(data):.0f}')
    ax.set_title(f'Image {label} Distribution', color='white')
    ax.set_xlabel(f'{label} (px)', color='#8b949e')
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')
    ax.legend(facecolor='#21262d', labelcolor='white')
plt.tight_layout()
plt.show()

## 5. Augmentation Preview

In [ ]:
visualize_augmentations(DATASET_ROOT)

## 6. Data Generator Test

In [ ]:
train_gen, valid_gen, test_gen = get_data_generators(DATASET_ROOT, batch_size=8)

batch_x, batch_y = next(train_gen)
print(f'Batch shape: {batch_x.shape}  Labels: {batch_y.astype(int)}')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.patch.set_facecolor('#0d1117')
for i, ax in enumerate(axes.flatten()):
    ax.imshow(batch_x[i])
    ax.set_title(CLASSES[int(batch_y[i])], color='#58a6ff')
    ax.axis('off')
fig.suptitle('Augmented Training Batch', color='white', fontsize=13)
plt.tight_layout()
plt.show()